# Modelo ARCH: Entrenamiento y Evaluación

In [2]:
import warnings
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.models.arch_wrapper import ARCHWrapper
from src.eval.scorer import VolatilityScorer
from src.trainer.walk_forward import WalkForwardTrainer

from src.utils import trainer_toolkit as tt
from src.utils import test_toolkit as tk

warnings.filterwarnings("ignore")

OUT_PATH = PROJECT_ROOT / "data" / "processed"
RESULTS_PATH = PROJECT_ROOT / "results" / "arch"
MODELS_PATH = PROJECT_ROOT / "models"

RESULTS_PATH.mkdir(parents=True, exist_ok=True)
MODELS_PATH.mkdir(parents=True, exist_ok=True)

df_train = pd.read_parquet(OUT_PATH / "brent_train.parquet")
y_train = df_train["log_returns"].values.astype("float64")
rs_train = df_train["rs_daily"].values.astype("float64")

df_test = pd.read_parquet(OUT_PATH / "brent_test.parquet")
y_test = df_test["log_returns"].values.astype("float64")
rs_test = df_test["rs_daily"].values.astype("float64")

print(f"Train: {len(y_train)} obs ({df_train.index[0].date()} → {df_train.index[-1].date()})")
print(f"Test:  {len(y_test)} obs ({df_test.index[0].date()} → {df_test.index[-1].date()})")
print(f"Resultados de tablas y gráficas: {RESULTS_PATH}")
print(f"Modelos entrenados (.joblib):    {MODELS_PATH}")

C:\Users\jcami\.conda\envs\brent-dl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train: 2171 obs (2014-01-03 → 2022-11-03)
Test:  953 obs (2022-11-07 → 2026-08-21)
Resultados de tablas y gráficas: C:\Users\jcami\OneDrive\Escritorio\deep_learning\brent-crude-nlp-market-dynamics\results\arch
Modelos entrenados (.joblib):    C:\Users\jcami\OneDrive\Escritorio\deep_learning\brent-crude-nlp-market-dynamics\models


## Introducción

Para modelar la dinámica de la volatilidad en series financieras como el crudo Brent, Robert F. Engle (1982) propuso el modelo Autorregresivo de Heterocedasticidad Condicional (ARCH). Esta formulación constituye un hito fundamental en la econometría, al permitir que la varianza condicional de los retornos dependa de la magnitud de los choques pasados, superando el supuesto de varianza constante.

Esta libreta tiene una doble función: por un lado, presentar, entrenar y evaluar el comportamiento empírico del modelo ARCH; y por otro, sentar las bases conceptuales y estructurales sobre las cuales se construirán y evaluarán modelos de volatilidad más avanzados en las siguientes secciones del proyecto.

### Fundamentos

Los retornos logarítmicos diarios se pueden representar como un proceso estocástico:

$$ r_t = \mu_t + \varepsilon_t, $$

donde $\mu_t$ denota la media condicional y $\varepsilon_t$ representa la innovación. Si se asume que la serie es estacionaria y su media está centrada en cero ($\mu_t = 0$), entonces $r_t = \varepsilon_t$. 

En su formulación clásica, las innovaciones siguen:

$$ \varepsilon_t = \sigma_t z_t, \quad z_t \sim i.i.d. \,\,\, \mathcal{N}(0,1), $$

con $\sigma_t^2$ denotando la varianza condicional y $z_t$ una variable aleatoria normal estándar. Bajo este modelo, la innovación $\varepsilon_t$ satisface:

$$ \mathbb{E}[\varepsilon_t \mid \mathcal{F}_{t-1}] = 0, \qquad \text{Var}(\varepsilon_t \mid \mathcal{F}_{t-1}) = \sigma_t^2, $$

es decir, la media condicional de la innovación es cero y su varianza condicional coincide con la varianza del proceso.

El supuesto de normalidad sobre $z_t$ puede relajarse. Como se evidenció en el EDA, los retornos del petroleo Brent presentan colas pesadas y asimetría, características que la distribución normal no captura adecuadamente. Por ello, es común reemplazar $\mathcal{N}(0,1)$ por distribuciones como la Student-$t$ o la Generalized Error Distribution (GED). En todos los casos, la distribución de $z_t$ se mantiene estandarizada, con media cero y varianza unitaria, de modo que la varianza condicional de $\varepsilon_t$ sigue siendo $\sigma_t^2$.

### Modelo ARCH

La varianza condicional se modela según el modelo ARCH de orden $q$:

$$ \sigma_t^2 = \omega + \sum_{i=1}^{q} \alpha_i \varepsilon_{t-i}^2, $$

donde $\omega > 0$ captura el nivel basal de varianza y los coeficientes $\alpha_i \geq 0$ cuantifican la influencia de las innovaciones al cuadrado rezagadas. Esta formulación constituye el punto de partida para la modelización de la volatilidad condicional y establece que la varianza de hoy depende de la magnitud de los choques pasados.

Para que el modelo sea válido, se requiere que la serie cumpla con la **condición de estacionariedad en covarianza**, la cual exige que $\sum_{i=1}^{q} \alpha_i < 1$. Esta condición garantiza que el impacto de un choque pasado se disipe con el tiempo (reversión a la media) y que la varianza no crezca indefinidamente. Bajo este supuesto, la varianza incondicional de largo plazo (el nivel medio al que tiende la serie) viene dada por $\mathbb{E}[\sigma_t^2] = \omega / (1 - \sum_{i=1}^{q} \alpha_i)$.

A lo largo de este trabajo, la implementación computacional del modelo ARCH y sus derivados econométricos se realizará utilizando la librería `arch` de Python en su versión `8.0.0`.

## Diseño Experimental

### Marco General

Para garantizar la comparabilidad entre las familias de modelos (econométricos, ML y DL), se establece una regla de paridad de información. En cualquier instante de tiempo $t$, todos los modelos tendrán acceso exclusivamente a los retornos logarítmicos históricos del crudo Brent (la información disponible en $\mathcal{F}_{t-1}$). Se excluye el uso de variables exógenas, el precio del activo y la volatilidad realizada durante la fase de ajuste, previniendo el sesgo de anticipación (*look-ahead bias*).

La variable objetivo para los 10 modelos es la volatilidad realizada. En la fase de evaluación, las predicciones de cada modelo se contrastan contra el estimador de volatilidad realizada de Rogers-Satchell (`rs_daily`), el cual actúa como proxy observable de la varianza. Dicha serie se utiliza para el cálculo de las métricas de error y funciones de pérdida aplicadas a todos los modelos.

:::{admonition} Información: Función de Pérdida QLIKE
:class: info

La función de pérdida QLIKE (por sus siglas en inglés *Quasi-Likelihood*) se emplea para evaluar pronósticos de volatilidad. Dados $\sigma_t^2$ como el proxy de volatilidad realizada y $\hat{\sigma}_t^2$ como la varianza condicional pronosticada, la función se define como:

$$
\text{QLIKE}(\sigma_t^2, \hat{\sigma}_t^2) = \frac{1}{T} \sum_{t=1}^{T} \left( \frac{\sigma_t^2}{\hat{\sigma}_t^2} - \ln\left(\frac{\sigma_t^2}{\hat{\sigma}_t^2}\right) - 1 \right)
$$

La función QLIKE presenta una estructura asimétrica, otorgando una mayor penalización a la subestimación de la volatilidad frente a su sobreestimación. De acuerdo con Patton (2011), el uso de esta función garantiza que el ordenamiento de los modelos no se altere por errores de medición en la proxy de volatilidad.
:::

La selección de la configuración óptima se realiza identificando la combinación de hiperparámetros que **minimiza** la pérdida QLIKE promedio. Es fundamental precisar que el cálculo de esta métrica se efectúa exclusivamente sobre las predicciones *out-of-fold*, es decir, sobre el bloque de validación de cada pliegue. Esto garantiza que la selección del modelo se base en su capacidad predictiva sobre datos no vistos durante la fase de estimación de parámetros, evitando el sobreajuste y la fuga de información.

### Entrenamiento y Validación Cruzada

El ajuste de los modelos y la búsqueda de hiperparámetros se realizarán mediante un esquema de validación cruzada de ventana expansiva (*walk-forward validation*). A diferencia de la validación cruzada tradicional, este método respeta la dependencia temporal de los datos: el conjunto de entrenamiento se expande progresivamente, mientras que el conjunto de validación permanece como el bloque temporal inmediatamente posterior.

Para garantizar un tamaño muestral mínimo y evitar estimaciones inestables, se establece un periodo de calentamiento (*warm-up*) inicial. Posteriormente, los datos se dividen en 5 pliegues cronológicos. En cada iteración, el modelo se estima sobre el bloque de entrenamiento y se evalúa con la pérdida QLIKE sobre el bloque de validación.

Dado que en los modelos econométricos la estimación de parámetros se realiza mediante Máxima Verosimilitud (MLE), un proceso determinista que no depende de inicializaciones aleatorias, la obtención de ejecuciones independientes requiere la alteración del punto de origen del *walk-forward*. Por lo tanto, se ejecutará el proceso 5 veces desplazando el inicio del *warm-up* para generar particiones temporales distintas. Los resultados de cada ejecución se almacenarán de forma individual, permitiendo el cálculo de la media y la desviación estándar de las métricas para cada configuración.

La selección de la configuración óptima se realiza identificando la combinación de hiperparámetros que **minimiza** la pérdida QLIKE promedio. Es fundamental precisar que el cálculo de esta métrica se efectúa exclusivamente sobre las predicciones *out-of-fold*, es decir, sobre el bloque de validación de cada pliegue. Esto asegura que la selección del modelo se base en su capacidad predictiva sobre datos no vistos durante la fase de estimación de parámetros, previniendo el sobreajuste y la fuga de información.

Una vez concluida la fase de validación y seleccionada la combinación de hiperparámetros que minimiza la pérdida QLIKE *out-of-fold*, se procede a una reestimación final del modelo. Dicha reestimación se realiza utilizando la totalidad del conjunto de entrenamiento disponible hasta el punto de corte temporal definido. 

## Entrenamiento

Para identificar los mejores hiperparámetros del modelo, se ejecutaron cinco iteraciones independientes del esquema de validación cruzada definido en experimentos anteriores. En cada iteración, se realizó una búsqueda en cuadrícula para el orden $q$, abarcando valores desde 1 hasta 10. El optimizador convergió en la mayoría de las configuraciones evaluadas. Se registró una advertencia de no convergencia para el orden $q=10$, derivada de un gradiente direccional positivo, lo que evidencia inestabilidad numérica asociada al incremento de la complejidad paramétrica.

In [4]:
scorer = VolatilityScorer()

for exec_id in range(1, 6):
    current_warmup = 1200 + (exec_id * 10) 
    exec_dir = RESULTS_PATH / f"exec_{exec_id}"
    exec_dir.mkdir(parents=True, exist_ok=True)
    
    trainer = WalkForwardTrainer(
        model_class=ARCHWrapper,
        param_grid={
            "q": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
            "dist": ["normal", "t", "ged"]
        },
        scorer=scorer,
        output_dir=exec_dir,
        n_splits=5,
        warmup=current_warmup
    )
    
    trainer.run(y_train, rs_train)

Grid Search:  30%|███       | 9/30 [00:06<00:18,  1.14config/s, params={'dist': 'normal', 'q': 10}]C:\Users\jcami\OneDrive\Escritorio\deep_learning\brent-crude-nlp-market-dynamics\src\models\arch_wrapper.py:90: ConvergenceWarning: The optimizer returned code 8. The message is:
Positive directional derivative for linesearch
See scipy.optimize.fmin_slsqp for code meaning.

  self.model_fit = am.fit(disp='off')
Grid Search: 100%|██████████| 30/30 [00:24<00:00,  1.22config/s, params={'dist': 'ged', 'q': 10}]   


Los resultados obtenidos en las cinco réplicas presentan una desviación estándar reducida, lo que evidencia la consistencia del proceso de validación. A partir de los resultados, se observa que las configuraciones con un mayor número de rezagos (valores elevados de $q$) tienden a situarse en las primeras posiciones, sugiriendo que una mayor memoria histórica favorece la minimización de la pérdida QLIKE en esta fase. La prevalencia de la distribución $t$ de Student en el mejor desempeño es congruente con las colas pesadas y la curtosis elevada documentadas previamente en el EDA.

In [2]:
tt.display_top_results(
    RESULTS_PATH, 
    top_n=5, 
    aggregate_mode="mean",
    column_names=(
        "Distribución", "Orden (q)", 
        "Réplica 1", "Réplica 2", "Réplica 3", "Réplica 4", "Réplica 5", 
        "Global (Media ± SD)"
    )
)

Distribución,Orden (q),Réplica 1,Réplica 2,Réplica 3,Réplica 4,Réplica 5,Global (Media ± SD)
t,10,0.461 ± 0.152,0.429 ± 0.088,0.432 ± 0.091,0.427 ± 0.085,0.459 ± 0.132,0.442 ± 0.017
t,8,0.459 ± 0.143,0.454 ± 0.129,0.449 ± 0.118,0.428 ± 0.073,0.428 ± 0.070,0.444 ± 0.015
ged,9,0.447 ± 0.110,0.442 ± 0.097,0.436 ± 0.082,0.456 ± 0.120,0.437 ± 0.082,0.444 ± 0.008
t,6,0.456 ± 0.138,0.453 ± 0.113,0.435 ± 0.068,0.431 ± 0.061,0.460 ± 0.112,0.447 ± 0.013
t,9,0.463 ± 0.144,0.463 ± 0.133,0.430 ± 0.072,0.458 ± 0.122,0.430 ± 0.067,0.449 ± 0.017


Para verificar la robustez de los resultados, la agregación se repitió utilizando la mediana y el rango intercuartílico (IQR). Se observa un cambio en el ordenamiento: el criterio de la mediana favorece configuraciones con un menor número de rezagos ($q$ entre 4 y 6) y las distribuciones Normal o $t$. Una hipótesis que explica esta discrepancia se fundamenta en la asimetría de la pérdida QLIKE y la naturaleza de los datos. La métrica de la media es altamente sensible a los eventos extremos; los modelos con mayor $q$ y distribución $t$ logran capturar con mayor precisión los días de alta volatilidad (colas pesadas), reduciendo el error promedio. En los días típicos de mercado (representados por la mediana), los modelos más parsimoniosos ofrecen pronósticos más ajustados, mientras que los modelos sobreparametrizados tienden a sobreestimar la volatilidad.

In [3]:
tt.display_top_results(
    RESULTS_PATH, 
    top_n=5, 
    aggregate_mode="median",
    column_names=(
        "Distribución", 
        "Orden (q)", 
        "Réplica 1", "Réplica 2", "Réplica 3", "Réplica 4", "Réplica 5", 
        "Global (Mediana ± IQR)"
    )
)

Distribución,Orden (q),Réplica 1,Réplica 2,Réplica 3,Réplica 4,Réplica 5,Global (Mediana ± IQR)
normal,6,0.416 ± 0.088,0.419 ± 0.092,0.424 ± 0.041,0.419 ± 0.011,0.426 ± 0.026,0.419 ± 0.005
t,5,0.420 ± 0.061,0.411 ± 0.113,0.418 ± 0.029,0.431 ± 0.005,0.429 ± 0.023,0.420 ± 0.012
normal,4,0.414 ± 0.067,0.418 ± 0.079,0.444 ± 0.026,0.421 ± 0.040,0.437 ± 0.017,0.421 ± 0.019
normal,5,0.411 ± 0.089,0.420 ± 0.094,0.425 ± 0.037,0.424 ± 0.033,0.423 ± 0.032,0.423 ± 0.005
ged,5,0.420 ± 0.076,0.425 ± 0.085,0.431 ± 0.029,0.431 ± 0.008,0.428 ± 0.065,0.428 ± 0.007


La representación gráfica de la evolución de la pérdida QLIKE en función del orden $q$ evidencia un pronunciado efecto de rendimientos decrecientes. Se observa una reducción abrupta de la métrica desde $q=1$ hasta $q=4$, seguida de un aplanamiento o "codo" en el intervalo comprendido entre $q=5$ y $q=6$. Este comportamiento respalda la hipótesis del punto de inflexión: incrementar la complejidad paramétrica más allá de este umbral no genera mejoras sustanciales en la capacidad predictiva.

In [6]:
tt.plot_metric_evolution(
    results_path=RESULTS_PATH,
    metric="qlike", 
    ylim=(0.40, 0.65),
    y_step=0.05, 
    aggregate_mode="mean",
    title="Evolución del QLIKE por Orden y Distribución",
    xlab="Orden del modelo (q)",
    ylab="Pérdida QLIKE (Media ± SE)",
    legend_title="Distribución",
    pads=(10, 10, 15), 
    label_size=11, 
    title_size=14
)

Al cuantificar las diferencias entre los órdenes $q=5$ y $q=6$, se observa que la variación en la pérdida QLIKE es marginal, con una diferencia máxima de 0.004 para la distribución Normal y una diferencia nula para la GED (0.428 frente a 0.429). La distribución $t$ de Student demuestra ser la opción más coherente, ya que logra un desempeño sobresaliente (0.420) con un menor número de parámetros. Aunque la distribución Normal ocupa el primer lugar bajo este criterio específico con $q=6$ (0.419), su desempeño general en la fase de validación fue consistentemente inferior, posicionándose como la peor elección en términos de QLIKE al requerir mayor complejidad para aproximarse a los resultados obtenidos por la distribución $t$.

In [5]:
tt.display_top_results(
    RESULTS_PATH, 
    q_filter=[5, 6],
    aggregate_mode="median",
    column_names=(
        "Distribución", 
        "Orden (q)", 
        "Réplica 1", "Réplica 2", "Réplica 3", "Réplica 4", "Réplica 5", 
        "Global (Mediana ± IQR)"
    )
)

Distribución,Orden (q),Réplica 1,Réplica 2,Réplica 3,Réplica 4,Réplica 5,Global (Mediana ± IQR)
normal,6,0.416 ± 0.088,0.419 ± 0.092,0.424 ± 0.041,0.419 ± 0.011,0.426 ± 0.026,0.419 ± 0.005
t,5,0.420 ± 0.061,0.411 ± 0.113,0.418 ± 0.029,0.431 ± 0.005,0.429 ± 0.023,0.420 ± 0.012
normal,5,0.411 ± 0.089,0.420 ± 0.094,0.425 ± 0.037,0.424 ± 0.033,0.423 ± 0.032,0.423 ± 0.005
ged,5,0.420 ± 0.076,0.425 ± 0.085,0.431 ± 0.029,0.431 ± 0.008,0.428 ± 0.065,0.428 ± 0.007
ged,6,0.413 ± 0.098,0.429 ± 0.092,0.436 ± 0.024,0.434 ± 0.009,0.422 ± 0.043,0.429 ± 0.012


La tabla de estimación revela que los coeficientes $\alpha$ presentan un valor idéntico de 0.1500. La suma total de estos coeficientes es de 0.75, satisfaciendo la condición de estacionariedad en covarianza al ser estrictamente menor que 1. La uniformidad exacta en los parámetros evidencia un problema de multicolinealidad. El optimizador distribuye el peso de manera equitativa entre los cinco rezagos al no lograr diferenciar la contribución individual de cada retorno pasado, lo que ilustra la falta de parsimonia del modelo ARCH.

In [8]:
best_model = ARCHWrapper(params={"q": 5, "dist": "t"})
best_model.fit(y_train)

df_params_boot = best_model.get_parameter_table(B=2000, decimals=4, block_size=22)

df_params_boot = df_params_boot.rename(columns={
    "Parameter": "Parámetro",
    "Coef": "Coeficiente",
    "Std. Error": "Error Estándar",
    "P-value": "p-valor",
    "CI Lower": "IC Inferior",
    "CI Upper": "IC Superior"
})

tt.display_styled_df(
    df=df_params_boot, 
    decimals=4, 
    p_value_cols=["p-valor"]
)

Parámetro,Coeficiente,Error Estándar,p-valor,IC Inferior,IC Superior
omega,0.0002,0.0000,< .001,0.0003,0.0005
alpha[1],0.1500,0.0384,< .001,0.0500,0.2659
alpha[2],0.1500,0.0373,< .001,0.0450,0.1354
alpha[3],0.1500,0.0383,< .001,0.0250,0.1232
alpha[4],0.1500,0.0391,< .001,0.0000,0.1200
alpha[5],0.1500,0.0382,< .001,0.0137,0.1200
nu,5.3418,0.8765,< .001,3.0257,5.3990


El parámetro $\nu$ (5.3418) define los grados de libertad de la distribución $t$. Un valor bajo y alejado del infinito confirma la presencia de colas pesadas. Este resultado valida empíricamente la elección de la distribución $t$ frente a la Normal. El hallazgo coincide directamente con las propiedades distributivas identificadas en el análisis exploratorio de datos.

Las pruebas de Ljung-Box y ARCH-LM confirman que el modelo capturó la estructura de dependencia temporal. No quedan patrones de autocorrelación ni efectos de heterocedasticidad condicional residuales en ninguno de los rezagos evaluados. El test BDS confirma la ausencia de efectos no lineales. El modelo agotó la información lineal y no lineal disponible en la serie.

In [9]:
std_residuals = best_model.model_fit.std_resid

df_diag = tt.run_model_diagnostics(
    std_residuals=std_residuals, 
    lags=(10, 20, 30), 
    p_adjust="bonferroni",
    alpha=0.05,
    column_titles=(
        "Prueba Estadística", 
        "Estadístico", "p-valor", 
        "p-ajustado (Bonferroni)", 
        "Conclusión (α = 0.05)"
    )
)

tt.display_styled_df(
    df=df_diag, decimals=3, 
    p_value_cols=["p-valor", "p-ajustado"]
)

Prueba Estadística,Estadístico,p-valor,p-ajustado (Bonferroni),Conclusión (α = 0.05)
Ljung-Box (Lag 10),6.440013,.777,1.000,Sin Autocorrelación
Ljung-Box (Lag 20),28.488359,.098,.295,Sin Autocorrelación
Ljung-Box (Lag 30),37.506037,.163,.489,Sin Autocorrelación
ARCH-LM (Lag 10),12.843029,.233,.698,Sin Efectos ARCH
ARCH-LM (Lag 20),25.642952,.178,.534,Sin Efectos ARCH
ARCH-LM (Lag 30),46.144563,.030,.090,Sin Efectos ARCH
Jarque-Bera,1923.678849,< .001,< .001,Residuos No Normales
Lilliefors,0.060391,< .001,.002,Residuos No Normales
BDS (Dim 2),0.3942651910543579,.693,.693,Sin Estructura No Lineal


La figura ilustra el ajuste in-sample del modelo ARCH(5) frente a la volatilidad realizada. La serie condicional reproduce la dinámica general durante los periodos de fluctuación moderada. El modelo captura los clústeres de volatilidad. Este comportamiento refleja la capacidad de asimilar la memoria de corto plazo a lo largo de la ventana temporal.

El comportamiento diverge durante el shock extremo de 2020. La volatilidad realizada registra un incremento abrupto e inusual. La volatilidad condicional se eleva, pero subestima la magnitud del evento, alcanzando apenas la mitad del nivel observado en el mercado. El modelo no logra capturar la intensidad de este shock incluso durante la fase de entrenamiento. Esta limitación refleja la naturaleza de reversión a la media del modelo ARCH. La arquitectura lineal no asimila la velocidad de las crisis sistémicas severas.

In [13]:
pred_vol_train = best_model.model_fit.conditional_volatility

tk.plot_volatility_fit(
    dates=df_train.index, 
    rs=rs_train, pred_vol=pred_vol_train,
    title="Volatilidad Realizada vs Condicional (Entrenamiento)",
    legend_size=13, legend_loc="upper left"
)

## Evaluación

La tabla presenta la comparación de las métricas de desempeño entre el conjunto de entrenamiento (In-Sample) y el de prueba (Out-of-Sample). La pérdida QLIKE registra un valor inferior en la fase de prueba (0.4112) frente al entrenamiento (0.4227). Las métricas de error puntual (MSE, RMSE y MAE) mantienen valores prácticamente idénticos, con diferencias absolutas en el orden de las diezmilésimas. Las métricas probabilísticas (CRPS y Pinball Loss en los cuantiles 0.05 y 0.95) también exhiben valores ligeramente inferiores en el conjunto de prueba. Esta consistencia numérica entre ambos conjuntos de datos evidencia la ausencia de sobreajuste. El modelo demuestra una capacidad de generalización estable al conservar su rendimiento predictivo sobre datos no vistos.

In [6]:
scorer = VolatilityScorer()

q_context = best_model.context_length
predictions_test = best_model.forecast(
    y_target=y_test, 
    initial_context=y_train[-q_context:]
)

tk.display_train_test_comparison(
    model=best_model, 
    scorer=scorer,
    rs_train=rs_train, 
    rs_test=rs_test,
    preds_test=predictions_test, 
    dist="t",                  
    nu=best_model.model_fit.params['nu'], 
    decimals=6, 
    column_titles=(
        "Métrica", 
        "Entrenamiento", 
        "Prueba",
        "Diferencia"
    ),
    show_diff=True
)

Métrica,Entrenamiento,Prueba,Diferencia
QLIKE,0.422704,0.411269,-0.011435
MSE,0.000004,0.000005,+0.000001
RMSE,0.001872,0.002149,+0.000276
MAE,0.000474,0.000478,+0.000004
CRPS,0.004763,0.004630,-0.000132
Pinball (τ=0.05),0.001863,0.001804,-0.000059
Pinball (τ=0.95),0.001802,0.001747,-0.000054


El desempeño predictivo es consistente con la fase de entrenamiento. La serie condicional reproduce la tendencia general de la volatilidad realizada durante los periodos de fluctuación moderada. Se mantiene la misma limitación estructural observada previamente. La volatilidad realizada registra un shock extremo a inicios de 2026, superando el valor de 0.25. La volatilidad condicional subestima la magnitud del evento al alcanzar un nivel cercano a 0.08. El patrón de ajuste in-sample se replica en la evaluación out-of-sample. Este resultado confirma la estabilidad predictiva del modelo y su incapacidad para asimilar saltos abruptos de volatilidad.

In [7]:
pred_vol_test = np.sqrt(predictions_test)

tk.plot_volatility_fit(
    dates=df_test.index, 
    rs=rs_test, 
    pred_vol=pred_vol_test, 
    title="Volatilidad Realizada vs Condicional (Prueba)",
    legend_loc="upper left",
    legend_size=13   
)